## Dynamic Interruption Demo

In [1]:
from langgraph.prebuilt import tools_condition, ToolNode
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState
from langgraph.types import interrupt, Command

In [2]:
def process(name : str, age : int) -> str:
    """Use this tool to process a person's admission request.
        You must provide:
        - name (str): Full name of the applicant
        - age (int): Age of the applicant
    """

    if age < 18:
        # Dynamic Interrupt
        approval = interrupt(f"Do you want to approve the admission of {name} who is {age} years old (minor)? (yes/no):")
        if approval is True:
             return  f"Processed the admission of {name}, who is minor! "
        else:
             return  f"Sorry! Cannot process the admission of {name}, who is a minor! "
    else:
        return  f"Processed the admission of {name}! "

In [3]:
# System message
sys_msg = SystemMessage(content="You are a helpful assistant. If the user asks to process admission, you MUST use the process tool.")

In [4]:
llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

In [5]:
# Create LLM that can call tools
tools = [process]
llm_with_tools = llm.bind_tools(tools)

In [6]:
# Node
def call_llm(state: MessagesState) -> MessagesState:
    ai_message = llm_with_tools.invoke([sys_msg] + state["messages"])
    return  {"messages": [ai_message]}

In [7]:
# Graph
builder = StateGraph(MessagesState)

# Define nodes: these do the work
builder.add_node("call_llm", call_llm)
builder.add_node("tools", ToolNode(tools))

# Define edges: these determine the control flow
builder.add_edge(START, "call_llm")
builder.add_conditional_edges(
    "call_llm",
    tools_condition
)

builder.add_edge("tools", "call_llm")
builder.add_edge("call_llm", END)

In [8]:
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

In [9]:
messages = [HumanMessage(
        content="Process admission of Scott Bentton who is 15 years old")]

In [10]:
# Thread
thread = {"configurable": {"thread_id": "1"}}
response = graph.invoke( {"messages" : messages}, thread)


In [11]:
print(response)

{'messages': [HumanMessage(content='Process admission of Scott Bentton who is 15 years old', additional_kwargs={}, response_metadata={}, id='c58d9fe6-05ea-4dea-a719-9677e10db2f2'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'process', 'arguments': '{"name": "Scott Bentton", "age": 15}'}, '__gemini_function_call_thought_signatures__': {'09dafe53-70dd-4acb-b3ca-3753a2bdc3bc': 'CqsCAWkUfRN0IIUcr44dBH7Q19rwRsAH649YEPoD8OJ9So4wShgDIdvIritHJxC7s72/7ovSiq3YqvutkMWSZqqWCZhl5ejIgg+V3TUOCuzAryQtR65NESk7k55dI5onySHVcpKYBiF2H/7W0nJWHg78ZfTiSaoaZtEvgjReDX4qUcNHrvqypWdT5LZ0V7wTruJyL6jvb7nAKlGIkkDjf8mRJ1OlxYYMdXSDp3/hxjPZ8doo8rZS/jLX6j+aGWdcFo/31v8o1vKTUpcwRxseYWE6EaSCLx+FeHnTxvHnCv6N82R5XeAqbtosaHljVwVfGvGtEdMEq+TaI6Lrfq3BP9HaJtUS8cWYFBF43Fh7a+7NtUx/ESscG1rIygja0PAbsadGa6YNyUbHtRY2hpg='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a187b8-c75c-7b03-b343-4b1de4973809-0', t

In [12]:
if "__interrupt__" in response:
    question = response['__interrupt__'][0].value   # Prompt
    user_approval = input(question)

    # Check approval and resume the graph with response from user
    response = graph.invoke(Command(resume=user_approval.lower() == "yes"), thread)


In [13]:
for m in response['messages']:
    m.pretty_print()

================================ Human Message =================================

Process admission of Scott Bentton who is 15 years old
================================== Ai Message ==================================
Tool Calls:
  process (09dafe53-70dd-4acb-b3ca-3753a2bdc3bc)
 Call ID: 09dafe53-70dd-4acb-b3ca-3753a2bdc3bc
  Args:
    name: Scott Bentton
    age: 15
================================= Tool Message =================================
Name: process

Processed the admission of Scott Bentton, who is minor! 
================================== Ai Message ==================================

The admission of Scott Bentton has been processed. He is a minor.
